# Decision Tree Intrusion Detection System (IDS)

## Overview
This notebook implements and evaluates a **Decision Tree classifier** for network intrusion detection using the NSL-KDD dataset.

### Dataset: NSL-KDD
- 41 features including network traffic characteristics
- Binary classification: Normal vs Attack
- Attack categories: DoS, Probe, R2L, U2R
- Train/Test split provided separately

### Methodology
1. **Data Loading & Exploration**: Load and analyze NSL-KDD dataset
2. **Data Preprocessing**: Handle categorical features, encoding
3. **Feature Engineering**: Feature selection, dimensionality reduction
4. **Model Training**: Decision Tree with hyperparameter tuning (GridSearchCV)
5. **Evaluation**: Cross-validation, confusion matrix, ROC curve, performance metrics

### Performance Metrics
- Accuracy
- Precision & Recall
- F1-Score
- ROC-AUC
- Confusion Matrix

## 1. Setup & Configuration

In [ ]:
# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
import random
import numpy as np
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# Import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import sklearn modules
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

In [ ]:
# Configuration
CONFIG = {
    'random_seed': 42,
    'feature_percentile': 30,
    'cv_folds': 10,
    'train_data_path': 'data/KDDTrain+.txt',
    'test_data_path': 'data/KDDTest+.txt',
}

## 2. Data Definition & Loading

In [ ]:
# NSL-KDD feature names
FEATURE_COLUMNS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'xAttack', 'level'
]

# Attack categories mapping
ATTACK_MAPPING = {
    'DoS': ['neptune', 'back', 'land', 'pod', 'smurf', 'teardrop', 'mailbomb', 'apache2', 'processtable', 'udpstorm', 'worm'],
    'R2L': ['ftp_write', 'guess_passwd', 'httptunnel', 'imap', 'multihop', 'named', 'phf', 'sendmail', 'snmpgetattack', 'snmpguess', 'spy', 'warezclient', 'warezmaster', 'xlock', 'xsnoop'],
    'Probe': ['ipsweep', 'mscan', 'nmap', 'portsweep', 'saint', 'satan'],
    'U2R': ['buffer_overflow', 'loadmodule', 'perl', 'ps', 'rootkit', 'sqlattack', 'xterm']
}

CATEGORICAL_FEATURES = ['protocol_type', 'service', 'flag']
TARGET_COLUMN = 'xAttack'
LEVEL_COLUMN = 'level'

In [ ]:
def load_nsl_kdd_data(train_path, test_path):
    """
    Load NSL-KDD dataset from CSV files.
    Creates synthetic data if files not found.
    """
    print("Loading NSL-KDD dataset...")
    try:
        df_train = pd.read_csv(train_path, header=None, names=FEATURE_COLUMNS)
        df_test = pd.read_csv(test_path, header=None, names=FEATURE_COLUMNS)
        print(f"✓ Training data shape: {df_train.shape}")
        print(f"✓ Testing data shape: {df_test.shape}")
        return df_train, df_test
    except FileNotFoundError:
        print(f"⚠ Warning: Data files not found at specified paths.")
        print(f"  Train path: {train_path}")
        print(f"  Test path: {test_path}")
        return None, None

# Load data
data_train, data_test = load_nsl_kdd_data(CONFIG['train_data_path'], CONFIG['test_data_path'])

# Handle case where data is not available
if data_train is None:
    print("\n⚠ Please download NSL-KDD dataset and place in 'data/' folder")
    print("Available at: https://www.unb.ca/cic/datasets/nsl-kdd.html")

## 3. Data Exploration & Preprocessing

In [ ]:
if data_train is not None:
    # Data quality checks
    print("=== DATA QUALITY CHECKS ===")
    print(f"Duplicates in training: {data_train.duplicated().sum()}")
    print(f"Duplicates in test: {data_test.duplicated().sum()}")
    print(f"Missing values in training: {data_train.isnull().sum().sum()}")
    print(f"Missing values in test: {data_test.isnull().sum().sum()}")
    print(f"\nTraining shape: {data_train.shape}")
    print(f"Test shape: {data_test.shape}")

In [ ]:
if data_train is not None:
    def consolidate_attack_labels(df):
        """
        Map individual attack types to attack categories (DoS, R2L, Probe, U2R).
        """
        df_copy = df.copy()
        for category, attacks in ATTACK_MAPPING.items():
            df_copy.loc[df_copy[TARGET_COLUMN].isin(attacks), TARGET_COLUMN] = category
        return df_copy
    
    data_train = consolidate_attack_labels(data_train)
    data_test = consolidate_attack_labels(data_test)
    
    print("\n=== ATTACK DISTRIBUTION ===")
    attack_dist = pd.concat(
        [data_train[TARGET_COLUMN].value_counts(), data_test[TARGET_COLUMN].value_counts()],
        axis=1, keys=['Train', 'Test']
    )
    print(attack_dist)

In [ ]:
if data_train is not None:
    # Visualize attack distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    data_train[TARGET_COLUMN].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Training Set: Attack Distribution', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Count')
    axes[0].set_xlabel('Attack Type')
    
    data_test[TARGET_COLUMN].value_counts().plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Test Set: Attack Distribution', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Count')
    axes[1].set_xlabel('Attack Type')
    
    plt.tight_layout()
    plt.show()

In [ ]:
if data_train is not None:
    # Separate features and target
    X_train = data_train.drop([TARGET_COLUMN, LEVEL_COLUMN], axis=1)
    y_train = data_train[[TARGET_COLUMN]]
    
    X_test = data_test.drop([TARGET_COLUMN, LEVEL_COLUMN], axis=1)
    y_test = data_test[[TARGET_COLUMN]]
    
    print("=== FEATURE-TARGET SPLIT ===")
    print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
    print(f"X_test: {X_test.shape}  | y_test: {y_test.shape}")

In [ ]:
if data_train is not None:
    def encode_categorical_features(X_train, X_test, categorical_cols):
        """
        Apply One-Hot encoding to categorical features.
        Ensures consistent encoding between train and test sets.
        """
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        
        X_train_encoded = encoder.fit_transform(X_train[categorical_cols])
        X_test_encoded = encoder.transform(X_test[categorical_cols])
        
        encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
        
        X_train_cat_df = pd.DataFrame(X_train_encoded, columns=encoded_feature_names, index=X_train.index)
        X_test_cat_df = pd.DataFrame(X_test_encoded, columns=encoded_feature_names, index=X_test.index)
        
        X_train = X_train.drop(categorical_cols, axis=1).reset_index(drop=True)
        X_test = X_test.drop(categorical_cols, axis=1).reset_index(drop=True)
        
        X_train_cat_df = X_train_cat_df.reset_index(drop=True)
        X_test_cat_df = X_test_cat_df.reset_index(drop=True)
        
        return pd.concat([X_train, X_train_cat_df], axis=1), pd.concat([X_test, X_test_cat_df], axis=1)
    
    X_train, X_test = encode_categorical_features(X_train, X_test, CATEGORICAL_FEATURES)
    print(f"\nAfter encoding:")
    print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

In [ ]:
if data_train is not None:
    def select_top_features(X_train, y_train, X_test, percentile=30):
        """
        Select top features using SelectPercentile with f_classif scoring.
        """
        y_train_flat = y_train.values.ravel() if hasattr(y_train, 'values') else y_train
        
        selector = SelectPercentile(f_classif, percentile=percentile)
        X_train_selected = selector.fit_transform(X_train, y_train_flat)
        X_test_selected = selector.transform(X_test)
        
        selected_features = X_train.columns[selector.get_support()]
        
        return (pd.DataFrame(X_train_selected, columns=selected_features),
                pd.DataFrame(X_test_selected, columns=selected_features))
    
    X_train_selected, X_test_selected = select_top_features(
        X_train, y_train, X_test, percentile=CONFIG['feature_percentile']
    )
    print(f"\n=== FEATURE SELECTION ===")
    print(f"Original features: {X_train.shape[1]} → Selected: {X_train_selected.shape[1]}")

## 4. Model Training & Evaluation

In [ ]:
if data_train is not None:
    def create_binary_classification_data(X_train, y_train, X_test, y_test, attack_type):
        """
        Create binary classification dataset: Normal vs specific attack type.
        """
        train_mask = (y_train[TARGET_COLUMN] == 'normal') | (y_train[TARGET_COLUMN] == attack_type)
        test_mask = (y_test[TARGET_COLUMN] == 'normal') | (y_test[TARGET_COLUMN] == attack_type)
        
        X_train_binary = X_train[train_mask].reset_index(drop=True)
        y_train_binary = y_train[train_mask].reset_index(drop=True)
        
        X_test_binary = X_test[test_mask].reset_index(drop=True)
        y_test_binary = y_test[test_mask].reset_index(drop=True)
        
        y_train_binary[TARGET_COLUMN] = (y_train_binary[TARGET_COLUMN] == attack_type).astype(int)
        y_test_binary[TARGET_COLUMN] = (y_test_binary[TARGET_COLUMN] == attack_type).astype(int)
        
        return X_train_binary, y_train_binary, X_test_binary, y_test_binary
    
    # Create datasets for each attack type
    attack_types = ['DoS', 'Probe', 'R2L', 'U2R']
    datasets = {}
    
    for attack in attack_types:
        X_tr, y_tr, X_te, y_te = create_binary_classification_data(
            X_train_selected, y_train, X_test_selected, y_test, attack
        )
        datasets[attack] = {'X_train': X_tr, 'y_train': y_tr, 'X_test': X_te, 'y_test': y_te}
        print(f"{attack}: Train {X_tr.shape[0]} | Test {X_te.shape[0]} | Attack ratio: {y_tr.mean().values[0]:.2%}")

In [ ]:
if data_train is not None:
    def train_and_evaluate_dt(X_train, y_train, X_test, y_test, attack_type):
        """
        Train Decision Tree with GridSearchCV and evaluate on test set.
        """
        print(f"\nTraining Decision Tree for {attack_type}...")
        
        y_train_flat = y_train.values.ravel() if hasattr(y_train, 'values') else y_train
        y_test_flat = y_test.values.ravel() if hasattr(y_test, 'values') else y_test
        
        # Hyperparameter tuning
        param_grid = {
            'max_depth': [5, 10, 15, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
        
        dt = DecisionTreeClassifier(random_state=CONFIG['random_seed'])
        grid_search = GridSearchCV(dt, param_grid, cv=5, scoring='f1', n_jobs=-1)
        grid_search.fit(X_train, y_train_flat)
        
        # Train final model
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
        y_pred_proba = best_model.predict_proba(X_test)[:, 1]
        
        # Calculate metrics
        metrics = {
            'Accuracy': accuracy_score(y_test_flat, y_pred),
            'Precision': precision_score(y_test_flat, y_pred),
            'Recall': recall_score(y_test_flat, y_pred),
            'F1-Score': f1_score(y_test_flat, y_pred),
            'ROC-AUC': auc(*roc_curve(y_test_flat, y_pred_proba)[:2])
        }
        
        return best_model, metrics, y_pred, y_pred_proba
    
    # Train models for all attack types
    models = {}
    results_list = []
    
    for attack in attack_types:
        model, metrics, y_pred, y_proba = train_and_evaluate_dt(
            datasets[attack]['X_train'], datasets[attack]['y_train'],
            datasets[attack]['X_test'], datasets[attack]['y_test'],
            attack
        )
        
        models[attack] = {'model': model, 'metrics': metrics, 'y_pred': y_pred, 'y_proba': y_proba}
        metrics['Attack'] = attack
        results_list.append(metrics)
    
    # Summary
    results_df = pd.DataFrame(results_list)[['Attack', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]
    print("\n" + "="*70)
    print("DECISION TREE IDS - PERFORMANCE SUMMARY")
    print("="*70)
    print(results_df.to_string(index=False))
    print("="*70)

## 5. Visualization & Analysis

In [ ]:
if data_train is not None:
    # Confusion Matrices
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    
    for idx, attack in enumerate(attack_types):
        y_test_flat = datasets[attack]['y_test'].values.ravel()
        y_pred = models[attack]['y_pred']
        cm = confusion_matrix(y_test_flat, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                    xticklabels=['Normal', attack], yticklabels=['Normal', attack], cbar=False)
        axes[idx].set_title(f'Confusion Matrix: {attack}', fontweight='bold')
        axes[idx].set_ylabel('True Label')
        axes[idx].set_xlabel('Predicted Label')
    
    plt.tight_layout()
    plt.show()

In [ ]:
if data_train is not None:
    # ROC Curves
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    
    for idx, attack in enumerate(attack_types):
        y_test_flat = datasets[attack]['y_test'].values.ravel()
        y_proba = models[attack]['y_proba']
        
        fpr, tpr, _ = roc_curve(y_test_flat, y_proba)
        roc_auc = auc(fpr, tpr)
        
        axes[idx].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC={roc_auc:.3f})')
        axes[idx].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
        axes[idx].set_xlim([0.0, 1.0])
        axes[idx].set_ylim([0.0, 1.05])
        axes[idx].set_xlabel('False Positive Rate')
        axes[idx].set_ylabel('True Positive Rate')
        axes[idx].set_title(f'ROC Curve: {attack}', fontweight='bold')
        axes[idx].legend(loc='lower right')
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if data_train is not None:
    # Performance comparison
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.ravel()
    
    metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    for idx, metric in enumerate(metrics_names):
        values = [models[attack]['metrics'][metric] for attack in attack_types]
        axes[idx].bar(attack_types, values, color=colors, edgecolor='black', linewidth=1.5)
        axes[idx].set_ylim([0, 1.1])
        axes[idx].set_title(f'{metric}', fontweight='bold')
        axes[idx].grid(True, axis='y', alpha=0.3)
    
    axes[-1].remove()
    plt.tight_layout()
    plt.show()

## 6. Conclusions

In [ ]:
if data_train is not None:
    print("\n" + "="*70)
    print("FINAL SUMMARY")
    print("="*70)
    
    print(f"\n📊 Dataset:")
    print(f"   Training: {len(y_train)} samples | Test: {len(y_test)} samples")
    print(f"   Features (original): 39 | Features (selected): {X_train_selected.shape[1]}")
    
    print(f"\n🎯 Best Classifier:")
    best_idx = results_df['F1-Score'].idxmax()
    best_row = results_df.loc[best_idx]
    print(f"   Attack Type: {best_row['Attack']}")
    print(f"   F1-Score: {best_row['F1-Score']:.4f}")
    print(f"   Accuracy: {best_row['Accuracy']:.4f}")
    
    print(f"\n📈 Average Performance:")
    print(f"   Accuracy: {results_df['Accuracy'].mean():.4f}")
    print(f"   Precision: {results_df['Precision'].mean():.4f}")
    print(f"   Recall: {results_df['Recall'].mean():.4f}")
    print(f"   F1-Score: {results_df['F1-Score'].mean():.4f}")
    print(f"   ROC-AUC: {results_df['ROC-AUC'].mean():.4f}")
    
    print("\n" + "="*70)